# Mock 3 (Variation): Reliability-Focused Incident Triage Agent

Timebox: **55 minutes**  
Language: **Python (Colab)**

## Progressive levels
1. Tool execution with validation
2. Add cache for repeated tool inputs
3. Add one retry for transient runtime failures
4. Return structured final output (`summary`, `action`, `confidence`)

## What to implement
- `execute_tool_call`
- `parse_final_output`
- `run_agent`


In [ ]:
import inspect
import json
from copy import deepcopy
from typing import Any, Callable

LOOKUP_ATTEMPTS: dict[str, int] = {}


def reset_state() -> None:
    LOOKUP_ATTEMPTS.clear()


def fetch_ticket(ticket_id: str) -> dict[str, Any]:
    return {
        "ticket_id": ticket_id,
        "service": "payments" if ticket_id == "inc-1" else "billing",
        "severity": "high",
    }


def lookup_runbook(service: str) -> dict[str, Any]:
    count = LOOKUP_ATTEMPTS.get(service, 0)
    LOOKUP_ATTEMPTS[service] = count + 1
    if service == "payments" and count == 0:
        raise RuntimeError("transient backend timeout")
    return {"service": service, "playbook": f"restart_{service}_workers"}


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "fetch_ticket": fetch_ticket,
    "lookup_runbook": lookup_runbook,
}


class TriageModel:
    def __init__(self, scenario: str) -> None:
        self.scenario = scenario
        self.step = 0

    def __call__(self, messages: list[dict[str, Any]]) -> dict[str, Any]:
        self.step += 1

        if self.scenario == "cache":
            if self.step == 1:
                return {
                    "stop_reason": "tool_use",
                    "tool_calls": [{"id": "a1", "name": "lookup_runbook", "input": {"service": "billing"}}],
                }
            if self.step == 2:
                return {
                    "stop_reason": "tool_use",
                    "tool_calls": [{"id": "a2", "name": "lookup_runbook", "input": {"service": "billing"}}],
                }
            return {
                "stop_reason": "end_turn",
                "output_text": json.dumps({
                    "summary": "billing issue mitigated",
                    "action": "restart_billing_workers",
                    "confidence": 0.78,
                }),
            }

        if self.scenario == "retry":
            if self.step == 1:
                return {
                    "stop_reason": "tool_use",
                    "tool_calls": [{"id": "b1", "name": "lookup_runbook", "input": {"service": "payments"}}],
                }
            return {
                "stop_reason": "end_turn",
                "output_text": json.dumps({
                    "summary": "payments issue mitigated",
                    "action": "restart_payments_workers",
                    "confidence": 0.81,
                }),
            }

        if self.scenario == "loop":
            return {
                "stop_reason": "tool_use",
                "tool_calls": [{"id": "loop", "name": "fetch_ticket", "input": {"ticket_id": "inc-1"}}],
            }

        return {"stop_reason": "end_turn", "output_text": "{}"}


In [ ]:
def execute_tool_call(
    tool_call: dict[str, Any],
    tool_registry: dict[str, Callable[..., Any]],
    cache: dict[str, dict[str, Any]],
) -> dict[str, Any]:
    """Execute a tool with cache + one retry for transient RuntimeError."""
    # TODO:
    # - Validate required fields and args
    # - Cache key = name + stable JSON of input
    # - If cached, return cached result with from_cache=True
    # - On RuntimeError containing 'transient', retry once
    # - Return tool message dict with is_error/content/from_cache
    raise NotImplementedError


def parse_final_output(output_text: str) -> dict[str, Any]:
    """Parse and validate structured final output with keys: summary, action, confidence."""
    # TODO: Parse JSON and validate required keys.
    raise NotImplementedError


def run_agent(
    user_prompt: str,
    model: Callable[[list[dict[str, Any]]], dict[str, Any]],
    tool_registry: dict[str, Callable[..., Any]],
    max_steps: int = 6,
) -> dict[str, Any]:
    """Progressive reliability loop: tool execution, cache tracking, retry behavior, structured final output."""
    # TODO: implement loop + stats (tool_calls, cache_hits)
    raise NotImplementedError


## Run tests
Run this test cell after implementing TODOs.


In [ ]:
def run_mock3_tests() -> None:
    reset_state()

    # 1) Cache behavior: same tool input should hit cache on second call
    model = TriageModel("cache")
    result = run_agent("triage billing", model, TOOL_REGISTRY)
    assert result["stats"]["cache_hits"] == 1
    assert LOOKUP_ATTEMPTS["billing"] == 1

    # 2) Retry behavior for transient errors
    reset_state()
    model = TriageModel("retry")
    result = run_agent("triage payments", model, TOOL_REGISTRY)
    assert LOOKUP_ATTEMPTS["payments"] == 2
    assert result["final"]["action"] == "restart_payments_workers"

    # 3) Structured final output required
    assert {"summary", "action", "confidence"}.issubset(result["final"].keys())

    # 4) Max steps defense
    model = TriageModel("loop")
    try:
        run_agent("loop", model, TOOL_REGISTRY, max_steps=3)
        raise AssertionError("Expected max_steps_exceeded")
    except RuntimeError as exc:
        assert "max_steps_exceeded" in str(exc)

    print("Mock 3 tests passed")


run_mock3_tests()
